# pairwise comparison + scaling

CGCoT breakdowns, async pairwise comparison, regularized Bradley-Terry (α = 0.01), simple and full bootstraps (1,000 iter). Theme validation and golden-pair tests live in `04_validation_tests.ipynb`.

In [ ]:
import os
REPO_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), '..')
    if os.path.basename(os.getcwd()) == 'code'
    else os.getcwd()
)
assert os.path.isdir(os.path.join(REPO_ROOT, 'pairwise-comparison')), 'bad REPO_ROOT'


In [ ]:
import os
import time
import random
import asyncio
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd

from openai import OpenAI, AsyncOpenAI
import choix


In [ ]:
#paths
INPUT_DIR              = os.path.join(REPO_ROOT, "pairwise-comparison/output")
BREAKDOWN_DIR          = os.path.join(INPUT_DIR, "full_breakdowns")
PAIRS_DIR              = os.path.join(INPUT_DIR, "pairs")
COMPARISON_RESULTS_DIR = os.path.join(INPUT_DIR, "comparison_results")
SCALING_DIR            = os.path.join(INPUT_DIR, "scaling")
REGULARIZED_DIR        = os.path.join(SCALING_DIR, "regularized")
FULL_BOOT_DIR          = os.path.join(SCALING_DIR, "regularized_full_bootstrap")
TIES_DIR               = os.path.join(SCALING_DIR, "regularized_with_ties")  # outputs consumed by 03_analysis.ipynb
for d in [BREAKDOWN_DIR, PAIRS_DIR, COMPARISON_RESULTS_DIR,
          REGULARIZED_DIR, FULL_BOOT_DIR, TIES_DIR]:
    os.makedirs(d, exist_ok=True)

#model + api
MODEL            = "gpt-4o-mini"
TEMPERATURE      = 0
MAX_TOKENS       = 300
MAX_SPEECH_CHARS = 4000
MAX_RETRIES      = 3
RATE_LIMIT_DELAY = 1

#pipeline
SEED                        = 42
SPEAKER_CAP                 = 3       
MIN_SPEECHES_PER_PARTY_YEAR = 5      
MIN_COMPARISONS_PER_SPEECH  = 10

#final four area-dimension combinations
DIMENSIONS = {
    "labour_market_policy__welfare_retrenchment":
        "labour_market_policy_welfare_retrenchment_breakdowns.csv",
    "education__social_investment":
        "education_social_investment_breakdowns.csv",
    "pension__fiscal_sustainability":
        "pension_fiscal_sustainability_breakdowns.csv",
    "family__social_investment":
        "family_social_investment_breakdowns.csv",
}
WELFARE_AREAS = ["labour_market_policy", "education", "pension", "family"]

random.seed(SEED)
np.random.seed(SEED)
client = OpenAI()


#government period map
GOVERNMENT_PERIODS = {
    2009: "Lokke I (2009-2011)", 2010: "Lokke I (2009-2011)",
    2011: "Thorning I (2011-2014)", 2012: "Thorning I (2011-2014)",
    2013: "Thorning I (2011-2014)",
    2014: "Thorning II (2014-2015)",
    2015: "Lokke II (2015-2016)",
    2016: "Lokke III (2016-2019)", 2017: "Lokke III (2016-2019)",
    2018: "Lokke III (2016-2019)",
    2019: "Frederiksen I (2019-2022)", 2020: "Frederiksen I (2019-2022)",
    2021: "Frederiksen I (2019-2022)", 2022: "Frederiksen I (2019-2022)",
    2023: "Frederiksen II (2022-)", 2024: "Frederiksen II (2022-)",
    2025: "Frederiksen II (2022-)",
}


## sampling speeches for breakdowns

In [ ]:
df = pd.read_csv(os.path.join(INPUT_DIR, "classified_speeches.csv"))

#keep only the four welfare areas retained after theme validation
df = df[df["final_llm_label"].isin(WELFARE_AREAS)].copy()

#drop parties whose (area x year) coverage is too sparse to track over time:
#>80% of cells fall below the 5-speech minimum.
all_combos = pd.MultiIndex.from_product(
    [df["party"].unique(), WELFARE_AREAS, df["year"].unique()],
    names=["party", "final_llm_label", "year"],
)
cell_n = (df.groupby(["party", "final_llm_label", "year"]).size()
            .reindex(all_combos, fill_value=0).rename("n").reset_index())
sparse = cell_n.groupby("party").apply(lambda g: (g["n"] < 5).mean())
keep_parties = sparse[sparse <= 0.80].index.tolist()
df = df[df["party"].isin(keep_parties)].copy()
print("parties retained:", keep_parties)
print(f"speeches after filter: {len(df):,}")


In [ ]:
#stratified sample with a speaker cap of 3 per (party, speaker, year)
#to mitigate dominance by individual spokespersons (paper, Breakdowns generation).

for area in WELFARE_AREAS:
    df_area = (df[df["final_llm_label"] == area]
                 .sort_values(["party", "speaker", "year", "speech_id"]))
    sampled = (df_area
               .groupby(["party", "speaker", "year"], group_keys=False)
               .apply(lambda g: g.sample(n=min(len(g), SPEAKER_CAP),
                                         random_state=SEED))
               .drop_duplicates("speech_id"))
    sampled.to_csv(os.path.join(INPUT_DIR, f"{area}_sample.csv"), index=False)
    print(f"{area}: {len(sampled):,} sampled")


## breakdown generation

In [ ]:
def get_breakdown(system_prompt, user_prompt, speech_text):
    """Generate a single concept-guided breakdown for one speech."""
    prompt = user_prompt.replace("{text}", speech_text[:MAX_SPEECH_CHARS])
    for attempt in range(MAX_RETRIES):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                messages=[{"role": "system", "content": system_prompt},
                          {"role": "user",   "content": prompt}],
            )
            content = r.choices[0].message.content.strip()
            return {"breakdown": content,
                    "has_position": "NO POSITION ON THIS DIMENSION" not in content.upper(),
                    "tokens_used": r.usage.total_tokens,
                    "status": "success"}
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                return {"breakdown": f"ERROR: {e}", "has_position": False,
                        "tokens_used": 0, "status": "error"}
            time.sleep((attempt + 1) * 5)

In [ ]:
#CGCoT breakdown prompts for the  final dimensions (Appendix B.5)

AREA_DIMENSIONS = {

    #labour market — welfare Retrenchment (merged at comparison stage)
    "labour_market_policy": {

        "conditional_welfare": {
            "name": "Conditional vs Unconditional Welfare Rights",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on welfare conditionality in labour market policy.

DIMENSION: Conditional vs Unconditional Welfare Rights
- POLE A (Unconditional): Welfare as a right to income security during unemployment, independent of job search behaviour or activation requirements
- POLE B (Conditional): Welfare tied to labour market participation, activation measures, work obligations, and sanctions for non-compliance""",

            "user_prompt": """Analyze this speech for positioning on CONDITIONAL VS UNCONDITIONAL welfare rights.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether welfare should be conditional on work/activation or unconditional?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about BEHAVIOURAL REQUIREMENTS — what the unemployed must DO to keep their benefits (job search, activation, accept work). It is NOT about the AMOUNT of benefits (that is generosity). It is NOT about WHO gets access based on nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech defends the right to receive benefits without behavioural conditions — no mandatory activation, job search requirements, or sanctions. The focus is on whether support is unconditional, not on the benefit amount.
- A Pole B speech argues that receiving benefits comes with obligations — the speaker frames unemployment support as a contract where the state provides income in exchange for active job search, participation in programmes, or acceptance of offered work. The focus is on what the recipient must DO, not on how much they receive.

BREAKDOWN:""",
            "pole_a": "Unconditional income security during unemployment",
            "pole_b": "Work-first, activation, sanctions for non-compliance",
        },

        "generosity_retrenchment": {
            "name": "Generous vs Residual Benefit Levels",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on welfare benefit generosity in labour market policy.

DIMENSION: Generous vs Residual Benefit Levels
- POLE A (Generous): Unemployment benefits as adequate income replacement that maintains dignified living standards
- POLE B (Residual): Benefits deliberately capped or reduced to ensure work always pays more than welfare""",

            "user_prompt": """Analyze this speech for positioning on BENEFIT GENEROSITY VS RETRENCHMENT.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether benefit levels should be generous/adequate or capped/reduced?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about the AMOUNT of benefits — whether levels are adequate or deliberately low. It is NOT about behavioural requirements for receiving benefits (that is conditionality). It is NOT about who qualifies based on nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech argues that benefit levels are too low or must be protected to ensure dignified living standards. The focus is on the AMOUNT — replacement rates, adequacy, or the real purchasing power of benefits — not on the conditions attached to receiving them.
- A Pole B speech argues that the level of benefits is too high relative to wages, creating a financial disincentive to work. The focus is on the AMOUNT of the benefit — caps, ceilings, or reductions in replacement rates — not on whether recipients must fulfil behavioural requirements.

BREAKDOWN:""",
            "pole_a": "Adequate benefit levels for dignified living",
            "pole_b": "Capped/reduced benefits to incentivize work",
        },
    },

    #education — social investment
    "education": {

        "social_investment": {
            "name": "Education as Equality vs Economic Investment",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on the purpose of education.

DIMENSION: Education as Social Equality vs Economic Investment
- POLE A (Equality): Education as a universal right promoting social equality, democratic citizenship and personal formation
- POLE B (Investment): Education framed as investment in productivity, labour supply and the future workforce""",

            "user_prompt": """Analyze this speech for positioning on EDUCATION AS EQUALITY VS ECONOMIC INVESTMENT.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether education primarily serves social equality/personal formation or economic productivity/labour supply?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about the PURPOSE of education — equality and personal formation vs economic productivity. It is NOT about behavioural requirements for receiving SU (that is conditionality). It is NOT about who should have access based on nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech treats education as valuable in itself — for personal growth, democratic participation, or social mobility. The speaker resists framing education in terms of labour market outcomes or economic returns.
- A Pole B speech frames education as a means to strengthen the workforce and national competitiveness. The speaker evaluates education by its labour market relevance, graduate employment rates, or contribution to economic growth.

BREAKDOWN:""",
            "pole_a": "Education for equality, citizenship and personal formation",
            "pole_b": "Education as investment in productivity and labour supply",
        },
    },

    #pension - fiscal sustainability
    "pension": {

        "fiscal_sustainability": {
            "name": "Stable Social Contract vs Fiscal Sustainability",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on pension policy.

DIMENSION: Stable Social Contract vs Fiscal Sustainability
- POLE A (Social contract): Retirement as a fixed, earned right at a stable predictable age
- POLE B (Fiscal sustainability): Pension reforms justified by demographic pressure, including linking retirement age to life expectancy""",

            "user_prompt": """Analyze this speech for positioning on STABLE PENSION RIGHTS VS FISCAL SUSTAINABILITY.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether pensions should be a stable right or adjusted for demographic sustainability?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

What each pole sounds like:
- A Pole A speech defends a fixed retirement age as a promise that workers have earned through a lifetime of work. The speaker may emphasise physical wear, fairness to manual workers, or the right to a dignified retirement without further extensions.
- A Pole B speech argues that retirement age must adapt to rising life expectancy and demographic realities. The speaker frames later retirement as necessary for fiscal sustainability and intergenerational fairness.

BREAKDOWN:""",
            "pole_a": "Retirement as a stable earned right at a fixed age",
            "pole_b": "Pension age adjusted for demographics and fiscal sustainability",
        },
    },

    #family - social investment
    "family": {

        "social_investment": {
            "name": "Family Support vs Human Capital Investment",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on the purpose of family policy.

DIMENSION: Family Support vs Human Capital Investment
- POLE A (Family support): Family policy as support for family life and child well-being
- POLE B (Investment): Early childhood policies framed as investment in human capital and future labour supply""",

            "user_prompt": """Analyze this speech for positioning on FAMILY SUPPORT VS HUMAN CAPITAL INVESTMENT.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether family policy primarily serves family well-being or future economic productivity?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

What each pole sounds like:
- A Pole A speech frames childcare and family policy around the well-being of children and parents — quality time, child development, and supporting family life as intrinsically valuable.
- A Pole B speech frames early childhood policies as investments that pay off later — through better educational outcomes, higher labour force participation, or a stronger future tax base. Children are discussed as future workers or human capital.

BREAKDOWN:""",
            "pole_a": "Family policy supports family life and child well-being",
            "pole_b": "Family policy as investment in human capital and labour supply",
        },
    },
}

In [ ]:
#generate breakdowns for areas

def run_breakdowns():
    total_calls = total_tokens = errors = 0

    for area, dims in AREA_DIMENSIONS.items():
        sample_path = os.path.join(INPUT_DIR, f"{area}_sample.csv")
        df = pd.read_csv(sample_path)

        for dim_key, dim in dims.items():
            out = os.path.join(BREAKDOWN_DIR, f"{area}_{dim_key}_breakdowns.csv")
            done_ids = (set(pd.read_csv(out)["speech_id"])
                        if os.path.exists(out) else set())
            new_rows = []

            text_col = next((c for c in ["text_clean", "text", "speech_text"]
                             if c in df.columns), None)

            for _, row in df.iterrows():
                sid = row["speech_id"]
                if sid in done_ids:
                    continue
                txt = row.get(text_col)
                if not isinstance(txt, str) or not txt.strip():
                    continue
                res = get_breakdown(dim["system_prompt"], dim["user_prompt"], txt)
                new_rows.append({
                    "speech_id": sid, "area": area, "dimension": dim_key,
                    "party": row.get("party"), "year": row.get("year"),
                    "speaker": row.get("speaker"),
                    "breakdown": res["breakdown"],
                    "has_position": res["has_position"],
                    "tokens_used": res["tokens_used"], "status": res["status"],
                })
                total_calls  += 1
                total_tokens += res["tokens_used"]
                errors       += res["status"] == "error"
                if total_calls % 200 == 0:
                    print(f"  [{total_calls}] {sid} | {total_tokens:,} tokens")
                time.sleep(RATE_LIMIT_DELAY)

            if new_rows:
                df_new = pd.DataFrame(new_rows)
                if os.path.exists(out):
                    df_new = pd.concat([pd.read_csv(out), df_new], ignore_index=True)
                df_new.to_csv(out, index=False)
                n_pos = df_new["has_position"].sum()
                print(f"  {area}/{dim_key}: {len(df_new)} done "
                      f"({100 * n_pos / len(df_new):.0f}% with position)")

    print(f"\nTotal calls: {total_calls} | tokens: {total_tokens:,} | errors: {errors}")


In [ ]:
run_breakdowns()

## pairwise sampling

In [ ]:
def load_dimension(filepath):
    d = pd.read_csv(filepath)
    d = d[d["has_position"]].copy()
    d["year"] = d["year"].astype(int)
    return d

def eligible_party_years(df):
    n = df.groupby(["party", "year"]).size()
    return {(p, y) for (p, y), c in n.items() if c >= MIN_SPEECHES_PER_PARTY_YEAR}

#randomized order 
def _ordered(a, b, meta):
    if random.random() > 0.5:
        a, b = b, a
    return {"speech_id_a": a, "speech_id_b": b,
            "party_a": meta[a]["party"], "party_b": meta[b]["party"],
            "year_a":  meta[a]["year"],  "year_b":  meta[b]["year"]}

#within-party cross-year + cross-party within-year sampling
def sample_pairs(df):
    eligible = eligible_party_years(df)
    meta = df.set_index("speech_id")[["party", "year"]].to_dict("index")
    pairs = []

    #within-party, year vs year+1 (temporal linkage)
    for party, g in df.groupby("party"):
        for y in sorted(g["year"].unique()):
            if (party, y) not in eligible or (party, y + 1) not in eligible:
                continue
            src = g[g["year"] == y]["speech_id"].tolist()
            tgt = g[g["year"] == y + 1]["speech_id"].tolist()
            for a in src:
                for b in random.sample(tgt, min(2, len(tgt))):
                    pairs.append(_ordered(a, b, meta))

    #cross-party, within-year
    for year, g in df.groupby("year"):
        parties = [p for p in g["party"].unique() if (p, year) in eligible]
        for pa, pb in combinations(parties, 2):
            sa = g[g["party"] == pa]["speech_id"].tolist()
            sb = g[g["party"] == pb]["speech_id"].tolist()
            for a in sa:
                for b in random.sample(sb, min(2, len(sb))):
                    pairs.append(_ordered(a, b, meta))

    #deduplicate
    seen, unique = set(), []
    for p in pairs:
        k = tuple(sorted([p["speech_id_a"], p["speech_id_b"]]))
        if k in seen:
            continue
        seen.add(k); unique.append(p)
    return unique, meta

#minimum speech matchups (10)
def topup(pairs, df, meta):
    counts = defaultdict(int)
    seen = set()
    for p in pairs:
        counts[p["speech_id_a"]] += 1
        counts[p["speech_id_b"]] += 1
        seen.add(tuple(sorted([p["speech_id_a"], p["speech_id_b"]])))

    extra = []
    for sid in [s for s in df["speech_id"] if counts[s] < MIN_COMPARISONS_PER_SPEECH]:
        info = meta[sid]
        needed = MIN_COMPARISONS_PER_SPEECH - counts[sid]
        cand = df[((df["party"] == info["party"]) &
                   (df["speech_id"] != sid) &
                   (df["year"].sub(info["year"]).abs() <= 4)) |
                  ((df["party"] != info["party"]) &
                   (df["year"].sub(info["year"]).abs() <= 1))]["speech_id"].tolist()
        cand = [c for c in cand if tuple(sorted([sid, c])) not in seen]
        for c in random.sample(cand, min(needed, len(cand))):
            seen.add(tuple(sorted([sid, c])))
            counts[sid] += 1; counts[c] += 1
            extra.append(_ordered(sid, c, meta))
    return pairs + extra

def attach_breakdowns(pairs, df):
    bm = df.set_index("speech_id")["breakdown"].to_dict()
    return pd.DataFrame([
        {**p, "breakdown_a": bm[p["speech_id_a"]],
              "breakdown_b": bm[p["speech_id_b"]]}
        for p in pairs if p["speech_id_a"] in bm and p["speech_id_b"] in bm])


In [ ]:
for dim_key, bd_file in DIMENSIONS.items():
    df_dim = load_dimension(os.path.join(BREAKDOWN_DIR, bd_file))
    pairs, meta = sample_pairs(df_dim)
    pairs = topup(pairs, df_dim, meta)
    pairs_df = attach_breakdowns(pairs, df_dim)
    pairs_df.to_csv(os.path.join(PAIRS_DIR, f"{dim_key}_pairs.csv"), index=False)
    print(f"{dim_key}: {len(pairs_df):,} pairs")

## pairwise comparison

In [ ]:
#two-step CGCoT comparison prompts - one config per dimension.
COMPARISON_PROMPTS = {
    f"{area}__{dim_key}": {
        "system_prompt": (
            "You are comparing two breakdowns of Danish parliamentary speeches "
            "on a single ideological dimension. Decide which speech expresses "
            "a position more strongly along the dimension. Reason about "
            "substance, conviction and framing - not surface vocabulary or order."
        ),
        "compare_direction": dim["compare_direction"],
    }
    for area, dims in AREA_DIMENSIONS.items()
    for dim_key, dim in dims.items()
}

def build_user_prompt(breakdown_a, breakdown_b, direction):
    return (f"SPEECH A:\n{breakdown_a}\n\n"
            f"SPEECH B:\n{breakdown_b}\n\n"
            f"TASK: Which speech is {direction}?\n"
            "Consider conviction, directness and framing - not just position direction.\n\n"
            "ANSWER one of:\n"
            f"  A     (Speech A is {direction})\n"
            f"  B     (Speech B is {direction})\n"
            "  EQUAL (both express similar positions)\n\n"
            "Provide your answer on the last line as: RESULT: [A/B/EQUAL]")


In [ ]:
async_client = AsyncOpenAI()

async def compare_pair(row, prompt_config, semaphore):
    async with semaphore:
        user_prompt = build_user_prompt(row["breakdown_a"], row["breakdown_b"],
                                        prompt_config["compare_direction"])
        for attempt in range(3):
            try:
                resp = await async_client.chat.completions.create(
                    model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                    messages=[{"role": "system",
                               "content": prompt_config["system_prompt"]},
                              {"role": "user", "content": user_prompt}])
                content = resp.choices[0].message.content
                result = "PARSE_ERROR"
                for line in reversed(content.strip().split("\n")):
                    if line.strip().upper().startswith("RESULT:"):
                        r = line.split(":", 1)[1].strip().upper()
                        if r in ("A", "B", "EQUAL"):
                            result = r
                        break
                return {**{k: row[k] for k in
                           ["speech_id_a","speech_id_b","party_a","party_b",
                            "year_a","year_b"]},
                        "result": result}
            except Exception:
                if attempt < 2:
                    await asyncio.sleep((attempt + 1) * 5)
        return {**{k: row.get(k, "") for k in
                   ["speech_id_a","speech_id_b","party_a","party_b",
                    "year_a","year_b"]}, "result": "ERROR"}

async def run_comparisons(max_concurrent=20, checkpoint_every=500):
    sem = asyncio.Semaphore(max_concurrent)
    for dim_key in DIMENSIONS:
        out = os.path.join(COMPARISON_RESULTS_DIR, f"{dim_key}_comparisons.csv")
        done = set()
        if os.path.exists(out):
            d = pd.read_csv(out)
            done = set(zip(d["speech_id_a"], d["speech_id_b"]))

        pairs = pd.read_csv(os.path.join(PAIRS_DIR, f"{dim_key}_pairs.csv"))
        todo = pairs[~pairs.apply(
            lambda r: (r["speech_id_a"], r["speech_id_b"]) in done, axis=1)]
        if todo.empty:
            print(f"{dim_key}: already complete"); continue
        print(f"{dim_key}: {len(todo)} pairs to run")

        prompt_config = COMPARISON_PROMPTS[dim_key]
        results = []
        rows = list(todo.iterrows())
        for i in range(0, len(rows), checkpoint_every):
            batch = rows[i:i + checkpoint_every]
            res = await asyncio.gather(*[compare_pair(r, prompt_config, sem)
                                         for _, r in batch])
            results.extend(res)
            df_save = pd.DataFrame(results)
            if os.path.exists(out):
                df_save = pd.concat([pd.read_csv(out), df_save], ignore_index=True)
            df_save.to_csv(out, index=False)
            print(f"  [{i + len(batch)}/{len(rows)}] checkpoint")


In [ ]:
await run_comparisons(max_concurrent=20)


In [ ]:
#distribution and positional-bias check (paper Table 5)
for dim_key in DIMENSIONS:
    d = pd.read_csv(os.path.join(COMPARISON_RESULTS_DIR,
                                 f"{dim_key}_comparisons.csv"))
    a  = (d["result"] == "A").sum()
    b  = (d["result"] == "B").sum()
    eq = (d["result"] == "EQUAL").sum()
    err = d["result"].isin(["ERROR", "PARSE_ERROR"]).sum()
    bias = 100 * a / (a + b) if (a + b) else 0
    print(f"{dim_key}: n={len(d):,} | A={a} B={b} EQUAL={eq} err={err} | "
          f"A-bias={bias:.1f}%")


## regularized Bradley-Terry + simple bootstrap

In [ ]:
#regularized alpha 0.01 found by senitive anlaysis with equal weight to equal pairs logic

def fit_regularized_bt(dim_key, bd_filename, alpha=0.01):
    df_comp = pd.read_csv(os.path.join(COMPARISON_RESULTS_DIR,
                                       f"{dim_key}_comparisons.csv"))
    df_comp = df_comp[df_comp["result"].isin(["A", "B", "EQUAL"])]

    #speech metadata
    df_meta = pd.read_csv(os.path.join(BREAKDOWN_DIR, bd_filename))
    df_meta = (df_meta[df_meta["has_position"]]
                 [["speech_id", "party", "year", "speaker"]].drop_duplicates())

    speeches = sorted(set(df_comp["speech_id_a"]) | set(df_comp["speech_id_b"]))
    idx = {s: i for i, s in enumerate(speeches)}

    comps = []
    for _, r in df_comp.iterrows():
        a, b = idx[r["speech_id_a"]], idx[r["speech_id_b"]]
        if r["result"] == "A":     comps.append((a, b))
        elif r["result"] == "B":   comps.append((b, a))
        else:                      comps += [(a, b), (b, a)]

    params = choix.opt_pairwise(len(speeches), comps, alpha=alpha)
    df_meta = df_meta.copy()
    df_meta["bt_raw"] = df_meta["speech_id"].map({s: params[i] for s, i in idx.items()})
    df_meta = df_meta.dropna(subset=["bt_raw"])
    df_meta["bt_score"] = ((df_meta["bt_raw"] - df_meta["bt_raw"].mean())
                            / df_meta["bt_raw"].std())
    df_meta["government"] = df_meta["year"].map(GOVERNMENT_PERIODS)
    df_meta["dim_key"]    = dim_key
    return df_meta[["speech_id", "dim_key", "party", "year", "speaker",
                    "government", "bt_score"]]


#simple bootstrap
def simple_bootstrap(df_speech, group_cols, n_bootstrap=1000, seed=SEED,
                     suffix="_simple"):
    rng = np.random.default_rng(seed)
    group_cols = list(group_cols)

    point = (df_speech.groupby(group_cols)
                      .agg(mean_score=("bt_score", "mean"),
                           n_speeches=("bt_score", "count"))
                      .reset_index())

    groups = {k: g["bt_score"].values
              for k, g in df_speech.groupby(group_cols)}
    boot = {k: [] for k in groups}
    for _ in range(n_bootstrap):
        for k, vals in groups.items():
            boot[k].append(rng.choice(vals, size=len(vals), replace=True).mean())

    ci = pd.DataFrame([
        {**dict(zip(group_cols, k if isinstance(k, tuple) else (k,))),
         f"ci_lower{suffix}": np.percentile(m, 2.5),
         f"ci_upper{suffix}": np.percentile(m, 97.5),
         f"ci_width{suffix}": np.percentile(m, 97.5) - np.percentile(m, 2.5)}
        for k, m in boot.items()
    ])
    return point.merge(ci, on=group_cols, how="left")


In [ ]:
#fit regularized BT once per dimension and aggregate to three levels:
#party-year, party-government, speaker-year. All consumed by 03_analysis.ipynb.

all_speech = []
all_py_simple, all_pg_simple, all_sy_simple = [], [], []

for dim_key, bd in DIMENSIONS.items():
    print(f"[BT + simple bootstrap] {dim_key}")
    df_speech = fit_regularized_bt(dim_key, bd, alpha=0.1)
    all_speech.append(df_speech)

    all_py_simple.append(simple_bootstrap(
        df_speech, ["party", "year"], n_bootstrap=1000).assign(dim_key=dim_key))
    all_pg_simple.append(simple_bootstrap(
        df_speech, ["party", "government"], n_bootstrap=1000).assign(dim_key=dim_key))
    all_sy_simple.append(simple_bootstrap(
        df_speech, ["party", "speaker", "year"], n_bootstrap=1000).assign(dim_key=dim_key))

#speech-level scores (resampled by 03 for downstream bootstraps)
df_speech_all = pd.concat(all_speech, ignore_index=True)
df_speech_all.to_csv(os.path.join(TIES_DIR,
                                  "simple_bootstrap_speech_level.csv"), index=False)

#aggregations with simple-bootstrap CIs
pd.concat(all_py_simple, ignore_index=True).to_csv(
    os.path.join(TIES_DIR, "simple_bootstrap_party_year.csv"), index=False)
pd.concat(all_pg_simple, ignore_index=True).to_csv(
    os.path.join(TIES_DIR, "simple_bootstrap_party_govt.csv"), index=False)
pd.concat(all_sy_simple, ignore_index=True).to_csv(
    os.path.join(TIES_DIR, "simple_bootstrap_speaker_year.csv"), index=False)
print("Saved speech-level + simple-bootstrap aggregations.")


## full bootstrap (comparison-level)

In [ ]:
#full bootstrap

def full_bootstrap_regularized(dim_key, bd_filename, alpha=0.1,
                                n_bootstrap=1000, seed=SEED,
                                group_cols_list=None):
    rng = np.random.default_rng(seed)
    if group_cols_list is None:
        group_cols_list = [("party", "year")]

    df_comp = pd.read_csv(os.path.join(COMPARISON_RESULTS_DIR,
                                       f"{dim_key}_comparisons.csv"))
    df_comp = df_comp[df_comp["result"].isin(["A", "B", "EQUAL"])].reset_index(drop=True)

    df_meta = pd.read_csv(os.path.join(BREAKDOWN_DIR, bd_filename))
    df_meta = (df_meta[df_meta["has_position"]]
                 [["speech_id", "party", "year", "speaker"]].drop_duplicates())
    df_meta["government"] = df_meta["year"].map(GOVERNMENT_PERIODS)

    speeches = sorted(set(df_comp["speech_id_a"]) | set(df_comp["speech_id_b"]))
    idx = {s: i for i, s in enumerate(speeches)}
    n_sp = len(speeches)

    def comps_from_rows(rows):
        out = []
        for r in rows:
            a, b = idx[r["speech_id_a"]], idx[r["speech_id_b"]]
            if r["result"] == "A":   out.append((a, b))
            elif r["result"] == "B": out.append((b, a))
            else:                    out += [(a, b), (b, a)]
        return out

    rows = df_comp.to_dict("records")

    params = choix.opt_pairwise(n_sp, comps_from_rows(rows), alpha=alpha)
    df_meta = df_meta[df_meta["speech_id"].isin(idx)].copy()
    df_meta["bt_raw"] = df_meta["speech_id"].map({s: params[i] for s, i in idx.items()})
    df_meta["bt_score"] = ((df_meta["bt_raw"] - df_meta["bt_raw"].mean())
                            / df_meta["bt_raw"].std())

    point_per_level = {
        tuple(g): (df_meta.groupby(list(g))
                          .agg(mean_score=("bt_score", "mean"),
                               n_speeches=("bt_score", "count"))
                          .reset_index())
        for g in group_cols_list
    }

    #bootstrap iterations 
    boot_per_level = {tuple(g): {} for g in group_cols_list}
    for it in range(n_bootstrap):
        sample = rng.integers(0, len(rows), size=len(rows))
        boot_comps = comps_from_rows([rows[i] for i in sample])
        try:
            p = choix.opt_pairwise(n_sp, boot_comps, alpha=alpha)
        except Exception:
            continue
        sc = np.array([p[idx[s]] for s in df_meta["speech_id"]])
        std = sc.std()
        if std == 0:
            continue
        z = (sc - sc.mean()) / std
        tmp = df_meta[["party", "year", "speaker", "government"]].copy()
        tmp["z"] = z
        for g in group_cols_list:
            agg = tmp.groupby(list(g))["z"].mean()
            for k, v in agg.items():
                key = k if isinstance(k, tuple) else (k,)
                boot_per_level[tuple(g)].setdefault(key, []).append(v)
        if (it + 1) % 200 == 0:
            print(f"  {it + 1}/{n_bootstrap}")

    out_per_level = {}
    for g in group_cols_list:
        gcols = list(g)
        ci = pd.DataFrame([
            {**dict(zip(gcols, k)),
             "ci_lower": np.percentile(m, 2.5),
             "ci_upper": np.percentile(m, 97.5),
             "ci_width": np.percentile(m, 97.5) - np.percentile(m, 2.5),
             "n_boot":   len(m)}
            for k, m in boot_per_level[tuple(g)].items() if len(m) >= 10
        ])
        out_per_level[tuple(g)] = (point_per_level[tuple(g)]
                                   .merge(ci, on=gcols, how="left"))
    return out_per_level


In [ ]:
#full bootstrap at the three aggregation levels named in the paper
#party-year, party-government, speaker-year.

LEVELS = [("party", "year"), ("party", "government"), ("party", "speaker", "year")]

all_py_full, all_pg_full, all_sy_full = [], [], []
for dim_key, bd in DIMENSIONS.items():
    print(f"[full bootstrap] {dim_key}")
    res = full_bootstrap_regularized(dim_key, bd, alpha=0.1,
                                     n_bootstrap=1000, group_cols_list=LEVELS)
    all_py_full.append(res[("party", "year")].assign(dim_key=dim_key))
    all_pg_full.append(res[("party", "government")].assign(dim_key=dim_key))
    all_sy_full.append(res[("party", "speaker", "year")].assign(dim_key=dim_key))

pd.concat(all_py_full, ignore_index=True).to_csv(
    os.path.join(TIES_DIR, "full_bootstrap_party_year.csv"), index=False)
pd.concat(all_pg_full, ignore_index=True).to_csv(
    os.path.join(TIES_DIR, "full_bootstrap_party_govt.csv"), index=False)
pd.concat(all_sy_full, ignore_index=True).to_csv(
    os.path.join(TIES_DIR, "full_bootstrap_speaker_year.csv"), index=False)
print("Saved full-bootstrap aggregations.")
